# Cross-fitted PainNAS on Google Colab

This notebook runs uncertainty-aware neural architecture search on five deterministic outer subject blocks. Each block search excludes the complete block, evaluates candidate architectures with three independent inner subject folds, and maximizes mean subject accuracy minus its standard error. The winning inner-fold checkpoint then warm-starts an individual LOSO continuation for every subject in that block. Each continuation uses a fresh optimizer and all other 86 subjects; the target subject is evaluated only on its predefined `Test` samples.

Select **Runtime → Change runtime type → GPU** before starting. Search databases, winning checkpoints, and completed LOSO folds are stored on Drive and can be resumed after a disconnect.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys
REPO_URL = 'https://github.com/hhihn/FewShotPainAdaptation.git'
PROJECT_DIR = Path('/content/FewShotPainAdaptation')
BRANCH_NAME = 'painnas'
if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only
%cd $PROJECT_DIR
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
assert (PROJECT_DIR / 'painnas/cross_fitted_loso.py').is_file()


## 2. Install pinned dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt


## 3. Stage BioVid on the local Colab SSD

In [ ]:
from data_loaders.dataset_staging import stage_predefined_dataset_from_archive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/PainData')
LOCAL_DATA_DIR = Path('/content/PainData')
BIOVID_ROOT = stage_predefined_dataset_from_archive(
    'biovid_part_a',
    drive_data_dir=DRIVE_DATA_DIR,
    local_data_dir=LOCAL_DATA_DIR,
    local_archive_dir=Path('/content'),
)
DATA_DIR = LOCAL_DATA_DIR
print('BioVid root:', BIOVID_ROOT)


## 4. Verify GPU and configure reproducibility

In [ ]:
import random
import numpy as np
import tensorflow as tf
GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Select a Colab GPU runtime.'
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', GPUS)


## 5. Configure cross-fitted NAS and LOSO

The full run performs five block-level NAS studies rather than 87 independent studies. Each trial is evaluated over three inner subject folds. The final continuation starts from the winning fold checkpoint but always creates a new optimizer. Change `RUN_NAME` whenever any configuration value changes; the manifest rejects incompatible resumes.

In [ ]:
from painnas.config import PainNASConfig
RUN_NAME = 'cross_fitted_run_001_late_binary'
FUSION_MODE = 'late'  # Set to 'late' for the multi-model late-fusion NAS.
CLASS_INDICES = '0,4'  # Comma-separated raw BioVid labels, e.g. '0,1,2,3,4'.
RAW_CLASS_IDS = tuple(int(value.strip()) for value in CLASS_INDICES.split(',') if value.strip())
if len(RAW_CLASS_IDS) < 2 or len(RAW_CLASS_IDS) != len(set(RAW_CLASS_IDS)):
    raise ValueError('CLASS_INDICES must contain at least two unique comma-separated integers.')
OUTPUT_DIR = Path('/content/drive/MyDrive/PainNAS') / RUN_NAME
CROSS_FITTED_DIR = OUTPUT_DIR / 'cross_fitted_loso'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = True
LOSO_START_INDEX = None  # One-based and inclusive.
LOSO_STOP_INDEX = None   # Set both values to run a resumable chunk.
MAX_FOLDS = None         # Debug-only cap after the index range.
CONFIG = PainNASConfig(
    seed=SEED,
    num_classes=len(RAW_CLASS_IDS),
    raw_class_ids=RAW_CLASS_IDS,
    batch_size=128,
    n_trials=10,
    search_max_epochs=30,
    cross_fitted_continuation_epochs=70,  # Maximum epochs; source-only early stopping may finish sooner.
    search_patience=8,
    outer_block_count=5,
    inner_fold_count=3,
    uncertainty_beta=1.0,
    max_parameters=40_000_000,
    bootstrap_samples=10_000,
    fusion_mode=FUSION_MODE,
)
print(CONFIG)
print('Output:', CROSS_FITTED_DIR)


## 6. Load BioVid and audit the deterministic subject plan

In [ ]:
import pandas as pd
from painnas.data import load_biovid_binary, build_cross_fitted_subject_plan
ARRAYS = load_biovid_binary(str(DATA_DIR), CONFIG)
SUBJECT_PLAN = build_cross_fitted_subject_plan(
    ARRAYS.unique_subjects,
    outer_block_count=CONFIG.outer_block_count,
    inner_fold_count=CONFIG.inner_fold_count,
    seed=CONFIG.seed,
)
plan_rows = []
for block_index, (block, inner_folds) in enumerate(
    zip(SUBJECT_PLAN.outer_blocks, SUBJECT_PLAN.inner_folds_by_block), start=1
):
    plan_rows.append({
        'outer_block': block_index,
        'outer_subject_count': len(block),
        'development_subject_count': sum(map(len, inner_folds)),
        'inner_fold_sizes': tuple(map(len, inner_folds)),
        'outer_subjects': block,
    })
display(pd.DataFrame(plan_rows))
assert sorted(s for block in SUBJECT_PLAN.outer_blocks for s in block) == sorted(ARRAYS.unique_subjects.tolist())
print('Samples:', len(ARRAYS.y), 'Shape:', ARRAYS.X.shape)


## 7. Inspect the fixed Table 2 baseline

In [ ]:
from painnas.model import ArchitectureSpec, LateFusionArchitectureSpec, build_model
BASELINE = LateFusionArchitectureSpec.baseline() if CONFIG.fusion_mode == 'late' else ArchitectureSpec.baseline()
BASELINE_MODEL = build_model(
    BASELINE,
    input_shape=(ARRAYS.num_modalities, ARRAYS.sequence_length, 1),
    num_classes=CONFIG.num_classes,
    modalities=CONFIG.modalities,
)
print(BASELINE)
print('Parameters:', f'{BASELINE_MODEL.count_params():,}')
del BASELINE_MODEL


## 8. Run or resume cross-fitted block NAS

For each required outer block, this runs or resumes one Optuna study, promotes its winning inner-fold checkpoint, and then performs warm-started individual LOSO continuations. A partial LOSO index range automatically reuses an existing block search when more subjects from that block are requested later.

## 9. Optional runtime cleanup

In [ ]:
# Release model resources, then disconnect and delete the hosted Colab runtime.
import gc
import logging
import sys

try:
    tf.keras.backend.clear_session()
except NameError:
    pass
gc.collect()
logging.shutdown()

try:
    from google.colab import runtime
except ImportError:
    print("Cleanup complete; no hosted Colab runtime was detected.")
else:
    print("Cleanup complete; disconnecting and deleting the Colab runtime.")
    sys.stdout.flush()
    sys.stderr.flush()
    runtime.unassign()